In [1]:
!pip install pytorchvideo --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 53.7 MB/s eta 0:00:00:00:0100:01
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=a40958f483648a8d2f69b0543b4d397373f4456391e04e9aba64be72c45f9a73
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=d1997b81b9b2303e32e90cd37c653db79e1fb6777aceed7d82e7e6de779742c1
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Crea

In [2]:
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import torch
import cv2
from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import Counter

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [3]:
# ──────────────────────────────────────────────
# PATHS
# ──────────────────────────────────────────────

XD_TRAIN  = "/kaggle/input/datasets/bypktt/xd-violence/train"
XD_TEST   = "/kaggle/input/datasets/bypktt/xd-violence/test"

RWF_TRAIN = "/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000/train"
RWF_TEST  = "/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000/val"

# Where to save the index CSVs
OUTPUT_DIR = "/kaggle/working"

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────

# Classes to SKIP from XD-Violence (irrelevant for mall surveillance)
XD_SKIP_CLASSES = {"caraccident", "explosion"}

# Normal class names per dataset (lowercase)
XD_NORMAL_LABEL  = "normal"
RWF_NORMAL_LABEL = "nonfight"

## Helpers

In [4]:
def scan_dataset(root_dir, normal_label, skip_classes=None, dataset_name=""):
    """
    Scans a dataset folder and returns a list of (filepath, label, class_name, dataset).
    label: 0 = Normal, 1 = Violence
    Skips any class in skip_classes (case-insensitive).
    """
    records = []
    skip_classes = {c.lower() for c in skip_classes} if skip_classes else set()

    if not os.path.exists(root_dir):
        print(f"  WARNING: Path not found: {root_dir}")
        return records

    classes = sorted(os.listdir(root_dir))

    for class_name in classes:
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        if class_name.lower() in skip_classes:
            print(f"  Skipping class: {class_name} ({dataset_name})")
            continue

        files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".mp4", ".avi", ".mkv", ".mov"))
        ]

        label = 0 if class_name.lower() == normal_label.lower() else 1

        for fname in files:
            records.append({
                "filepath":    os.path.join(class_path, fname),
                "label":       label,
                "class_name":  class_name,
                "dataset":     dataset_name,
            })

    return records


def print_summary(df, split_name):
    """Prints a detailed breakdown of a dataset split."""
    print(f"\n{'='*55}")
    print(f"  {split_name}")
    print(f"{'='*55}")

    total    = len(df)
    normal   = (df["label"] == 0).sum()
    violence = (df["label"] == 1).sum()

    print(f"  Total videos : {total}")
    print(f"  Normal       : {normal}  ({normal/total*100:.1f}%)")
    print(f"  Violence     : {violence}  ({violence/total*100:.1f}%)")

    print(f"\n  Breakdown by class:")
    for (dataset, class_name), grp in df.groupby(["dataset", "class_name"]):
        label_str = "Normal" if grp["label"].iloc[0] == 0 else "Violence"
        print(f"    [{dataset}]  {class_name:<18} {len(grp):>5} files  →  {label_str}")

    print(f"{'='*55}")


def build_sampler(labels):
    """
    Builds a WeightedRandomSampler that balances Normal vs Violence per batch.
    Does NOT remove any samples — just adjusts sampling probability.
    """
    label_counts = Counter(labels)
    total        = len(labels)

    # Weight for each sample = 1 / count of its class
    class_weights = {cls: total / count for cls, count in label_counts.items()}
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float),
        num_samples = total,
        replacement = True,
    )

    print(f"\n  Sampler class weights:")
    for cls, w in class_weights.items():
        label_str = "Normal" if cls == 0 else "Violence"
        print(f"    {label_str}: {w:.4f}")

    return sampler

## BUILD TRAIN SPLIT

In [5]:
print("\nScanning datasets...")
print("\n[TRAIN]")

xd_train_records  = scan_dataset(
    XD_TRAIN,
    normal_label = XD_NORMAL_LABEL,
    skip_classes = XD_SKIP_CLASSES,
    dataset_name = "XD-Violence",
)

rwf_train_records = scan_dataset(
    RWF_TRAIN,
    normal_label = RWF_NORMAL_LABEL,
    skip_classes = None,
    dataset_name = "RWF-2000",
)

train_df = pd.DataFrame(xd_train_records + rwf_train_records)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle order


Scanning datasets...

[TRAIN]
  Skipping class: CarAccident (XD-Violence)
  Skipping class: Explosion (XD-Violence)


## BUILD TEST SPLIT

In [6]:
print("\n[TEST]")

xd_test_records  = scan_dataset(
    XD_TEST,
    normal_label = XD_NORMAL_LABEL,
    skip_classes = XD_SKIP_CLASSES,
    dataset_name = "XD-Violence",
)

rwf_test_records = scan_dataset(
    RWF_TEST,
    normal_label = RWF_NORMAL_LABEL,
    skip_classes = None,
    dataset_name = "RWF-2000",
)

test_df = pd.DataFrame(xd_test_records + rwf_test_records)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


[TEST]
  Skipping class: CarAccident (XD-Violence)
  Skipping class: Explosion (XD-Violence)


## Summaries, Sampler, And Index csv

In [7]:
# ──────────────────────────────────────────────
# PRINT SUMMARIES
# ──────────────────────────────────────────────

print_summary(train_df, "TRAIN SPLIT")
print_summary(test_df,  "TEST SPLIT")

# ──────────────────────────────────────────────
# BUILD SAMPLER FOR TRAINING
# ──────────────────────────────────────────────

print("\n[WeightedRandomSampler — Train]")
sampler = build_sampler(train_df["label"].tolist())

# ──────────────────────────────────────────────
# SAVE INDEX CSVs
# ──────────────────────────────────────────────

train_csv = os.path.join(OUTPUT_DIR, "train_index.csv")
test_csv  = os.path.join(OUTPUT_DIR, "test_index.csv")

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv,   index=False)

print(f"\n  Saved train index → {train_csv}")
print(f"  Saved test  index → {test_csv}")


  TRAIN SPLIT
  Total videos : 4663
  Normal       : 2849  (61.1%)
  Violence     : 1814  (38.9%)

  Breakdown by class:
    [RWF-2000]  Fight                800 files  →  Violence
    [RWF-2000]  NonFight             800 files  →  Normal
    [XD-Violence]  Abuse                 31 files  →  Violence
    [XD-Violence]  Fighting             377 files  →  Violence
    [XD-Violence]  Normal              2049 files  →  Normal
    [XD-Violence]  Riot                 374 files  →  Violence
    [XD-Violence]  Shooting             232 files  →  Violence

  TEST SPLIT
  Total videos : 975
  Normal       : 500  (51.3%)
  Violence     : 475  (48.7%)

  Breakdown by class:
    [RWF-2000]  Fight                200 files  →  Violence
    [RWF-2000]  NonFight             200 files  →  Normal
    [XD-Violence]  Abuse                  7 files  →  Violence
    [XD-Violence]  Fighting             107 files  →  Violence
    [XD-Violence]  Normal               300 files  →  Normal
    [XD-Violence]  Riot 

## FINAL BALANCE CHECK

In [8]:
print("\n[Final Balance Check]")
train_normal   = (train_df["label"] == 0).sum()
train_violence = (train_df["label"] == 1).sum()
test_normal    = (test_df["label"] == 0).sum()
test_violence  = (test_df["label"] == 1).sum()

print(f"  Train → Normal: {train_normal}  Violence: {train_violence}  "
      f"Ratio: {train_normal/(train_normal+train_violence)*100:.1f}/{train_violence/(train_normal+train_violence)*100:.1f}")
print(f"  Test  → Normal: {test_normal}   Violence: {test_violence}   "
      f"Ratio: {test_normal/(test_normal+test_violence)*100:.1f}/{test_violence/(test_normal+test_violence)*100:.1f}")

print("\nDataset preparation complete.")


[Final Balance Check]
  Train → Normal: 2849  Violence: 1814  Ratio: 61.1/38.9
  Test  → Normal: 500   Violence: 475   Ratio: 51.3/48.7

Dataset preparation complete.


---

---

# Model preparation and training

In [9]:
!pip install gdown

import gdown

file_id = "1WD0mwpOkqfYlJr0r7Lss2puGefsCF_Ms"
url = f"https://drive.google.com/uc?id={file_id}"

output = "latest_checkpoint_slowfastEpoch5.pth"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1WD0mwpOkqfYlJr0r7Lss2puGefsCF_Ms
From (redirected): https://drive.google.com/uc?id=1WD0mwpOkqfYlJr0r7Lss2puGefsCF_Ms&confirm=t&uuid=c7b9bca6-84ac-469a-a405-83cbab54d001
To: /kaggle/working/latest_checkpoint_slowfastEpoch5.pth
100%|██████████| 142M/142M [00:00<00:00, 144MB/s]  


'latest_checkpoint_slowfastEpoch5.pth'

## Config

In [10]:
import torch

# Fix the checkpoint in-place — strip module. prefix and re-save
ckpt_path = "/kaggle/working/latest_checkpoint_slowfastEpoch5.pth"
fixed_path = "/kaggle/working/latest_checkpoint_slowfast_fixed.pth"

ckpt = torch.load(ckpt_path, map_location="cpu")

state_dict = ckpt["model_state_dict"]

if any(k.startswith("module.") for k in state_dict.keys()):
    print("Stripping module. prefix...")
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    ckpt["model_state_dict"] = state_dict
    print("Done.")

torch.save(ckpt, fixed_path)
print(f"Fixed checkpoint saved to: {fixed_path}")

Stripping module. prefix...
Done.
Fixed checkpoint saved to: /kaggle/working/latest_checkpoint_slowfast_fixed.pth


In [11]:
TRAIN_CSV = "/kaggle/working/train_index.csv"
TEST_CSV  = "/kaggle/working/test_index.csv"

IMG_SIZE        = 224
NUM_FRAMES_FAST = 32   # Fast pathway samples — high temporal resolution
ALPHA           = 4    # Slow pathway samples NUM_FRAMES_FAST / ALPHA frames
NUM_FRAMES_SLOW = NUM_FRAMES_FAST // ALPHA   # = 8

BATCH_SIZE = 4   # SlowFast is heavier than VideoMAE — start conservative
EPOCHS     = 20

LR_HEAD     = 3e-4
LR_BACKBONE = 1e-5
UNFREEZE_EPOCH = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_SAVE_PATH  = "/kaggle/working/best_slowfast.pth"
LATEST_CKPT_SAVE_PATH = "/kaggle/working/latest_checkpoint_slowfast.pth"

# RESUME_CHECKPOINT_PATH = None
RESUME_CHECKPOINT_PATH = "/kaggle/working/latest_checkpoint_slowfast_fixed.pth"

EARLY_STOPPING_PATIENCE = 5

# Kinetics-400 normalization stats (what SlowFast was pretrained with)
MEAN = [0.45, 0.45, 0.45]
STD  = [0.225, 0.225, 0.225]

## DATASET

In [12]:
class SlowFastVideoDataset(Dataset):
    """
    Loads videos from a CSV index built by prepare_dataset.py.
    Returns frames sampled at NUM_FRAMES_FAST temporal resolution.
    The slow pathway subsampling happens in collate_fn.
    """

    def __init__(self, csv_path):
        self.df      = pd.read_csv(csv_path)
        self.samples = self.df["filepath"].tolist()
        self.labels  = self.df["label"].tolist()

        print(f"  Loaded {len(self.samples)} videos from {csv_path}")
        counts = Counter(self.labels)
        print(f"  Normal: {counts[0]}  Violence: {counts[1]}")

    def load_video(self, path):
        cap          = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames       = []

        if total_frames <= 0:
            cap.release()
            return [
                np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                for _ in range(NUM_FRAMES_FAST)
            ]

        if total_frames < NUM_FRAMES_FAST:
            indices  = list(range(total_frames))
            indices += [total_frames - 1] * (NUM_FRAMES_FAST - total_frames)
        else:
            indices = np.linspace(0, total_frames - 1, NUM_FRAMES_FAST).astype(int)

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()

            if not ret:
                frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()
        return frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frames = self.load_video(self.samples[idx])
        label  = self.labels[idx]

        # Convert to tensor: (T, H, W, C) -> (C, T, H, W)
        video_tensor = torch.from_numpy(np.stack(frames)).float() / 255.0
        video_tensor = video_tensor.permute(3, 0, 1, 2)

        # Normalize per Kinetics stats
        mean = torch.tensor(MEAN).view(3, 1, 1, 1)
        std  = torch.tensor(STD).view(3, 1, 1, 1)
        video_tensor = (video_tensor - mean) / std

        return video_tensor, label


def collate_fn(batch):
    """
    Builds the [slow_pathway, fast_pathway] input SlowFast expects.
    fast_pathway: all NUM_FRAMES_FAST frames  -> (B, C, 32, H, W)
    slow_pathway: every ALPHA-th frame        -> (B, C, 8,  H, W)
    """
    videos = torch.stack([item[0] for item in batch])   # (B, C, T, H, W)
    labels = torch.tensor([item[1] for item in batch])

    fast_pathway = videos
    slow_indices = torch.linspace(
        0, videos.shape[2] - 1, NUM_FRAMES_SLOW
    ).long()
    slow_pathway = videos[:, :, slow_indices, :, :]

    return [slow_pathway, fast_pathway], labels


def build_sampler(labels):
    counts         = Counter(labels)
    total          = len(labels)
    class_weights  = {cls: total / count for cls, count in counts.items()}
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float),
        num_samples = total,
        replacement = True,
    )

## MODEL

In [13]:
class SlowFastClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        # Load pretrained SlowFast from PyTorchVideo (Kinetics-400)
        self.backbone = torch.hub.load(
            "facebookresearch/pytorchvideo",
            "slowfast_r50",
            pretrained=True,
        )

        # The original head outputs 400 Kinetics classes.
        # Replace it with a binary classifier.
        # slowfast_r50's head is at .blocks[-1] (ResNetBasicHead)
        in_features = self.backbone.blocks[-1].proj.in_features

        self.backbone.blocks[-1].proj = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2),
        )

    def forward(self, pathways):
        return self.backbone(pathways)

## OPTIMIZER / SCHEDULER HELPERS

In [14]:
def get_backbone_params(model):
    """All params except the final classifier head."""
    head_params = set(model.backbone.blocks[-1].proj.parameters())
    return [p for p in model.parameters() if p not in head_params]


def get_head_params(model):
    return list(model.backbone.blocks[-1].proj.parameters())


def build_optimizer_and_scheduler(model, backbone_unfrozen, epochs_remaining):
    if backbone_unfrozen:
        optimizer = optim.AdamW([
            {"params": get_backbone_params(model),
             "lr": LR_BACKBONE, "weight_decay": 0.01},
            {"params": get_head_params(model),
             "lr": LR_HEAD / 3, "weight_decay": 0.001},
        ])
    else:
        optimizer = optim.AdamW(
            get_head_params(model),
            lr=LR_HEAD,
            weight_decay=0.001,
        )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs_remaining, 1), eta_min=1e-6
    )
    return optimizer, scheduler

## CHECKPOINT HELPERS

In [15]:
def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    best_f1, patience, backbone_unfrozen):
    # Always unwrap DataParallel before saving
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    state_dict = raw_model.state_dict()

    # Double-check no module. prefix sneaked in
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    torch.save({
        "epoch":                epoch,
        "model_state_dict":     state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_f1":              best_f1,
        "patience":             patience,
        "backbone_unfrozen":    backbone_unfrozen,
    }, path)
    print(f"  >> Checkpoint saved → {path}")


def load_checkpoint(path, model, device):
    print(f"\nResuming from: {path}")
    ckpt = torch.load(path, map_location=device)

    state_dict = ckpt["model_state_dict"]

    # Strip 'module.' prefix if checkpoint was saved with DataParallel
    if any(k.startswith("module.") for k in state_dict.keys()):
        print("  Stripping 'module.' prefix from checkpoint keys...")
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    return ckpt

## TRAINING

In [16]:
def train():
    print(f"\nUsing device: {DEVICE}")

    print("\nLoading datasets...")
    train_dataset = SlowFastVideoDataset(TRAIN_CSV)
    test_dataset  = SlowFastVideoDataset(TEST_CSV)

    sampler = build_sampler(train_dataset.labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size  = BATCH_SIZE,
        sampler     = sampler,
        collate_fn  = collate_fn,
        num_workers = 4,
        pin_memory  = True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        collate_fn  = collate_fn,
        num_workers = 4,
        pin_memory  = True,
    )

    model = SlowFastClassifier().to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    # Helper to access underlying model regardless of DataParallel wrapping
    def base_model():
        return model.module if isinstance(model, nn.DataParallel) else model

    START_EPOCH       = 0
    best_f1           = 0.0
    patience          = 0
    backbone_unfrozen = False

    # Freeze everything except the head initially
    for param in base_model().parameters():
        param.requires_grad = False
    for param in get_head_params(base_model()):
        param.requires_grad = True

    optimizer, scheduler = build_optimizer_and_scheduler(
        base_model(), backbone_unfrozen=False, epochs_remaining=EPOCHS
    )

    if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
        print(f"\nResuming from: {RESUME_CHECKPOINT_PATH}")
        ckpt = torch.load(RESUME_CHECKPOINT_PATH, map_location=DEVICE)

        base_model().load_state_dict(ckpt["model_state_dict"])

        START_EPOCH       = ckpt["epoch"] + 1
        best_f1           = ckpt["best_f1"]
        patience          = ckpt["patience"]
        backbone_unfrozen = ckpt.get("backbone_unfrozen", False)

        if backbone_unfrozen:
            for param in base_model().parameters():
                param.requires_grad = True

        optimizer, scheduler = build_optimizer_and_scheduler(
            base_model(), backbone_unfrozen, EPOCHS - START_EPOCH
        )
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        print(f"Resumed at epoch {START_EPOCH + 1}/{EPOCHS}")
        print(f"Best F1 so far: {best_f1:.4f}")
    else:
        print("No checkpoint found — starting fresh.")

    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler()

    for epoch in range(START_EPOCH, EPOCHS):

        if not backbone_unfrozen and epoch >= UNFREEZE_EPOCH:
            print(f"\n>>> Unfreezing backbone at epoch {epoch + 1}")
            for param in base_model().parameters():
                param.requires_grad = True
            backbone_unfrozen = True
            optimizer, scheduler = build_optimizer_and_scheduler(
                base_model(), True, EPOCHS - epoch
            )

        print(f"\nEpoch {epoch + 1}/{EPOCHS} "
              f"[backbone {'unfrozen' if backbone_unfrozen else 'frozen'}]")

        model.train()
        train_loss = 0.0

        for pathways, labels in tqdm(train_loader, desc="Training"):
            pathways = [p.to(DEVICE) for p in pathways]
            labels   = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(pathways)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        scheduler.step()

        model.eval()
        epoch_preds  = []
        epoch_labels = []

        with torch.no_grad():
            for pathways, labels in tqdm(test_loader, desc="Evaluating"):
                pathways = [p.to(DEVICE) for p in pathways]

                with torch.cuda.amp.autocast():
                    outputs = model(pathways)

                preds = outputs.argmax(dim=1)
                epoch_preds.extend(preds.cpu().numpy())
                epoch_labels.extend(labels.numpy())

        acc       = accuracy_score(epoch_labels, epoch_preds)
        precision = precision_score(epoch_labels, epoch_preds, zero_division=0)
        recall    = recall_score(epoch_labels, epoch_preds, zero_division=0)
        f1        = f1_score(epoch_labels, epoch_preds, zero_division=0)
        cm        = confusion_matrix(epoch_labels, epoch_preds)
        avg_loss  = train_loss / len(train_loader)

        print(f"\nLoss      : {avg_loss:.4f}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"F1        : {f1:.4f}")
        print(f"Confusion Matrix:\n{cm}")

        save_checkpoint(
            LATEST_CKPT_SAVE_PATH,
            epoch, model, optimizer, scheduler,
            best_f1, patience, backbone_unfrozen,
        )

        if f1 > best_f1:
            best_f1  = f1
            patience = 0
            model_state = base_model().state_dict()
            torch.save(model_state, BEST_MODEL_SAVE_PATH)
            print(f"  >> Best model saved (F1={best_f1:.4f})")
        else:
            patience += 1
            print(f"  No improvement. Patience: {patience}/{EARLY_STOPPING_PATIENCE}")
            if patience >= EARLY_STOPPING_PATIENCE:
                print("\nEarly stopping triggered.")
                break

    print("\n" + "="*55)
    print("BEST MODEL — FINAL EVALUATION")
    print("="*55)

    base_model().load_state_dict(
        torch.load(BEST_MODEL_SAVE_PATH, map_location=DEVICE)
    )
    model.eval()

    final_preds  = []
    final_labels = []

    with torch.no_grad():
        for pathways, labels in tqdm(test_loader, desc="Final Eval"):
            pathways = [p.to(DEVICE) for p in pathways]
            with torch.cuda.amp.autocast():
                outputs = model(pathways)
            preds = outputs.argmax(dim=1)
            final_preds.extend(preds.cpu().numpy())
            final_labels.extend(labels.numpy())

    print(f"Accuracy  : {accuracy_score(final_labels, final_preds):.4f}")
    print(f"Precision : {precision_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"F1        : {f1_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(final_labels, final_preds)}")

## INFERENCE

In [17]:
def extract_frames(path, num_frames=NUM_FRAMES_FAST, img_size=IMG_SIZE):
    cap          = cv2.VideoCapture(path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames       = []

    if total_frames <= 0:
        cap.release()
        return [np.zeros((img_size, img_size, 3), dtype=np.uint8)
                for _ in range(num_frames)]

    if total_frames < num_frames:
        indices  = list(range(total_frames))
        indices += [total_frames - 1] * (num_frames - total_frames)
    else:
        indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames


def load_model_for_inference(weights_path=BEST_MODEL_SAVE_PATH):
    model = SlowFastClassifier()
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("Model loaded successfully.")
    return model


def predict_video(video_path, model, threshold=0.4):
    print(f"\nProcessing: {video_path}")

    frames = extract_frames(video_path)

    video_tensor = torch.from_numpy(np.stack(frames)).float() / 255.0
    video_tensor = video_tensor.permute(3, 0, 1, 2)  # (C, T, H, W)

    mean = torch.tensor(MEAN).view(3, 1, 1, 1)
    std  = torch.tensor(STD).view(3, 1, 1, 1)
    video_tensor = (video_tensor - mean) / std
    video_tensor = video_tensor.unsqueeze(0)  # add batch dim -> (1, C, T, H, W)

    fast_pathway = video_tensor.to(DEVICE)
    slow_indices = torch.linspace(0, video_tensor.shape[2] - 1, NUM_FRAMES_SLOW).long()
    slow_pathway = video_tensor[:, :, slow_indices, :, :].to(DEVICE)

    with torch.no_grad():
        logits = model([slow_pathway, fast_pathway])
        probs  = torch.softmax(logits, dim=1)[0]

    violence_conf = probs[1].item()
    prediction    = "Violence / Anomaly" if violence_conf >= threshold else "Normal"

    print(f"Confidence : Normal={probs[0]:.2%}  Violence={probs[1]:.2%}")
    print(f"Prediction : --> {prediction} <--")
    return prediction, probs.cpu().numpy()

## ENTRY POINT(TRAINING)

In [18]:
if __name__ == "__main__":

    train()


Using device: cuda

Loading datasets...
  Loaded 4663 videos from /kaggle/working/train_index.csv
  Normal: 2849  Violence: 1814
  Loaded 975 videos from /kaggle/working/test_index.csv
  Normal: 500  Violence: 475
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOWFAST_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/SLOWFAST_8x8_R50.pyth


100%|██████████| 264M/264M [00:00<00:00, 398MB/s] 


Using 2 GPUs

Resuming from: /kaggle/working/latest_checkpoint_slowfast_fixed.pth


/tmp/ipykernel_58/1848414568.py:79: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


Resumed at epoch 6/20
Best F1 so far: 0.8279

>>> Unfreezing backbone at epoch 6

Epoch 6/20 [backbone unfrozen]


...


Loss      : 0.2448
Accuracy  : 0.8636
Precision : 0.9071
Recall    : 0.8021
F1        : 0.8514
Confusion Matrix:
[[461  39]
 [ 94 381]]
  >> Checkpoint saved → /kaggle/working/latest_checkpoint_slowfast.pth
  >> Best model saved (F1=0.8514)

Epoch 9/20 [backbone unfrozen]


Training:   0%|          | 0/1166 [00:00<?, ?it/s]/tmp/ipykernel_58/1848414568.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  60%|██████    | 701/1166 [49:58<18:50,  2.43s/it]  [mov,mp4,m4a,3gp,3g2,mj2 @ 0x2f3b1800] Referenced QT chapter track not found
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x239a4200] Referenced QT chapter track not found
Evaluating:   0%|          | 0/244 [00:00<?, ?it/s]/tmp/ipykernel_58/1848414568.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Evaluating: 100%|██████████| 244/244 [16:55<00:00,  4.16s/it]



Loss      : 0.1376
Accuracy  : 0.8421
Precision : 0.9235
Recall    : 0.7368
F1        : 0.8197
Confusion Matrix:
[[471  29]
 [125 350]]
  >> Checkpoint saved → /kaggle/working/latest_checkpoint_slowfast.pth
  No improvement. Patience: 1/5

Epoch 10/20 [backbone unfrozen]


Training:   0%|          | 0/1166 [00:00<?, ?it/s]/tmp/ipykernel_58/1848414568.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  18%|█▊        | 214/1166 [15:17<1:08:01,  4.29s/it]


KeyboardInterrupt: 

## ENTRY POINT(INFERENCE)

In [ ]:
# 

## ZIPPING MODELS

In [ ]:
!zip outputs.zip best_model_combined.pth latest_checkpoint_combined.pth